# LeetCode 914 — X of a Kind in a Deck of Cards bằng Python

**Link đề:** https://leetcode.com/problems/x-of-a-kind-in-a-deck-of-cards/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Counting, Math, Number Theory (ước chung lớn nhất — GCD)

```python
class Solution:
    def hasGroupsSizeX(self, deck: list[int]) -> bool:
        ...
```

## Mục tiêu học
1. Đọc đúng đề: "chia bộ bài thành các nhóm cùng kích thước `X > 1`, mỗi nhóm chỉ một loại số".
2. Biết **đếm tần suất** bằng `dict` và `collections.Counter`.
3. Viết lời giải trực giác: thử mọi `X` từ 2 trở lên.
4. Nhận ra insight toán học: đáp án là `True` khi và chỉ khi **ƯCLN (gcd) của mọi tần suất ≥ 2**.
5. Hiểu thuật toán Euclid tính `gcd`, dùng `math.gcd` và `functools.reduce`.
6. Trace từng bước, chứng minh tính đúng, phân tích Time/Space, làm bài tập biến thể.

**Cách học:** đọc markdown → đoán output cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — chia bộ bài thành các nhóm bằng nhau

Cho mảng số nguyên `deck`, `deck[i]` là số ghi trên lá bài thứ `i`.

Hãy chia **toàn bộ** các lá bài thành **một hoặc nhiều nhóm** sao cho:
- Mỗi nhóm có **đúng `X` lá**, với `X > 1` (cùng một `X` cho mọi nhóm).
- Mọi lá trong cùng một nhóm ghi **cùng một số**.

Trả về `True` nếu chia được, ngược lại `False`.

| `deck` | Output | Giải thích |
|---|:---:|---|
| `[1,2,3,4,4,3,2,1]` | `True` | chia `[1,1],[2,2],[3,3],[4,4]` với `X = 2` |
| `[1,1,1,2,2,2,3,3]` | `False` | số 1, 2 có 3 lá, số 3 có 2 lá → không có `X` chung |
| `[1]` | `False` | chỉ 1 lá, mà `X` phải > 1 |
| `[1,1,2,2,2,2]` | `True` | `X = 2`: `[1,1],[2,2],[2,2]` — một số có thể chiếm **nhiều nhóm** |

**Constraints:** `1 <= deck.length <= 10^4`, `0 <= deck[i] < 10^4`.

Câu hỏi tự đặt trước khi code:
- Thứ tự các lá bài có quan trọng không? (Không — ta được tự do xếp nhóm.)
- Thứ duy nhất quan trọng là gì? → **mỗi số xuất hiện bao nhiêu lần**.
- Một số xuất hiện 4 lần có thể chia thành nhóm kích thước 2 không? (Có: 2 nhóm.)

In [1]:
# Nhìn ví dụ 4: số 2 xuất hiện 4 lần, chia thành 2 nhóm kích thước 2
deck = [1, 1, 2, 2, 2, 2]
X = 2
groups = []
for value in sorted(set(deck)):
    cnt = deck.count(value)
    for _ in range(cnt // X):
        groups.append([value] * X)
print("Các nhóm:", groups)

Các nhóm: [[1, 1], [2, 2], [2, 2]]


## 2. Nền tảng Python cần dùng

Bài này cần 5 viên gạch:
1. `dict` để đếm tần suất (tự đếm bằng tay).
2. `collections.Counter` — công cụ đếm có sẵn.
3. Toán tử chia lấy dư `%` và khái niệm **ước số**.
4. **ƯCLN / gcd** và thuật toán Euclid.
5. `all(...)`, `math.gcd`, `functools.reduce`.

### 2.1. Đếm tần suất bằng `dict`

`dict` lưu cặp **key → value**. Ở đây key là số trên lá bài, value là số lần xuất hiện.

- `counts.get(x, 0)` đọc value của `x`, nếu chưa có thì trả về `0`.
- `counts[x] = ...` ghi/ghi đè value.

In [2]:
deck = [1, 2, 3, 4, 4, 3, 2, 1]
counts = {}
for x in deck:
    counts[x] = counts.get(x, 0) + 1
    print(f"đọc {x} -> counts = {counts}")
print("Kết quả:", counts)
print("Chỉ lấy các tần suất:", list(counts.values()))

đọc 1 -> counts = {1: 1}
đọc 2 -> counts = {1: 1, 2: 1}
đọc 3 -> counts = {1: 1, 2: 1, 3: 1}
đọc 4 -> counts = {1: 1, 2: 1, 3: 1, 4: 1}
đọc 4 -> counts = {1: 1, 2: 1, 3: 1, 4: 2}
đọc 3 -> counts = {1: 1, 2: 1, 3: 2, 4: 2}
đọc 2 -> counts = {1: 1, 2: 2, 3: 2, 4: 2}
đọc 1 -> counts = {1: 2, 2: 2, 3: 2, 4: 2}
Kết quả: {1: 2, 2: 2, 3: 2, 4: 2}
Chỉ lấy các tần suất: [2, 2, 2, 2]


### 2.2. `collections.Counter` — đếm trong một dòng

`Counter` là một lớp con của `dict`, chuyên để đếm. `Counter(deck)` làm đúng việc vòng lặp ở trên.

- `Counter(...).values()` cho các tần suất.
- `Counter(...).most_common()` sắp theo tần suất giảm dần (hữu ích khi debug).
- Truy cập key không tồn tại trả về `0` thay vì lỗi `KeyError`.

In [3]:
from collections import Counter

c = Counter([1, 1, 1, 2, 2, 2, 3, 3])
print(c)
print("values:", list(c.values()))
print("most_common:", c.most_common())
print("c[99] =", c[99], "(không lỗi, trả 0)")

Counter({1: 3, 2: 3, 3: 2})
values: [3, 3, 2]
most_common: [(1, 3), (2, 3), (3, 2)]
c[99] = 0 (không lỗi, trả 0)


### 2.3. Phép chia lấy dư `%` và khái niệm ước số

`a % b` là **số dư** khi chia `a` cho `b`.
- `6 % 2 == 0` → 2 là **ước** của 6 (6 chia hết cho 2).
- `6 % 4 == 2` → 4 không phải ước của 6.

Liên hệ với bài: một số xuất hiện `cnt` lần chia được thành các nhóm kích thước `X` **khi và chỉ khi** `cnt % X == 0`.

In [4]:
for cnt in [4, 6, 3]:
    for X in [2, 3, 4]:
        ok = cnt % X == 0
        print(f"cnt={cnt}, X={X}: cnt % X = {cnt % X} -> {'chia được' if ok else 'không chia được'}")
    print()

cnt=4, X=2: cnt % X = 0 -> chia được
cnt=4, X=3: cnt % X = 1 -> không chia được
cnt=4, X=4: cnt % X = 0 -> chia được

cnt=6, X=2: cnt % X = 0 -> chia được
cnt=6, X=3: cnt % X = 0 -> chia được
cnt=6, X=4: cnt % X = 2 -> không chia được

cnt=3, X=2: cnt % X = 1 -> không chia được
cnt=3, X=3: cnt % X = 0 -> chia được
cnt=3, X=4: cnt % X = 3 -> không chia được



### 2.4. `all(...)` — "mọi phần tử đều thỏa"

`all(iterable)` trả `True` nếu **mọi** phần tử đều đúng (và `True` cho iterable rỗng).
Ta viết điều kiện dạng **generator expression**: `all(cnt % X == 0 for cnt in counts)`.

Đọc là: "với mọi `cnt` trong `counts`, `cnt % X == 0`".

In [5]:
counts = [4, 6, 2]
print(all(cnt % 2 == 0 for cnt in counts))   # True: 4,6,2 đều chẵn
print(all(cnt % 3 == 0 for cnt in counts))   # False: 4 không chia hết cho 3
print(all([]))                               # True với danh sách rỗng

True
False
True


### 2.5. ƯCLN (gcd) và thuật toán Euclid

**Ước chung lớn nhất** (greatest common divisor, `gcd`) của `a` và `b` là số lớn nhất chia hết cả hai.
- `gcd(4, 6) = 2`, `gcd(3, 2) = 1`, `gcd(6, 9) = 3`.

**Thuật toán Euclid** dựa trên tính chất: `gcd(a, b) = gcd(b, a % b)`, và `gcd(a, 0) = a`.

Ví dụ `gcd(48, 18)`:
```text
gcd(48, 18) -> 48 % 18 = 12 -> gcd(18, 12)
gcd(18, 12) -> 18 % 12 = 6  -> gcd(12, 6)
gcd(12, 6)  -> 12 % 6  = 0  -> gcd(6, 0) = 6
```

Vì sao đúng? Mọi ước chung của `a` và `b` cũng chia hết `a - k*b = a % b`, và ngược lại. Nên tập ước chung không đổi khi thay `(a, b)` bằng `(b, a % b)`.

In [6]:
def my_gcd(a: int, b: int) -> int:
    while b != 0:
        print(f"  gcd({a}, {b}) -> {a} % {b} = {a % b}")
        a, b = b, a % b
    return a

print("Kết quả:", my_gcd(48, 18))
print("Kết quả:", my_gcd(3, 2))

  gcd(48, 18) -> 48 % 18 = 12
  gcd(18, 12) -> 18 % 12 = 6
  gcd(12, 6) -> 12 % 6 = 0
Kết quả: 6
  gcd(3, 2) -> 3 % 2 = 1
  gcd(2, 1) -> 2 % 1 = 0
Kết quả: 1


Cú pháp `a, b = b, a % b` là **gán đồng thời** (tuple unpacking): vế phải được tính xong **trước**, rồi mới gán cho `a` và `b`. Nhờ vậy không cần biến tạm.

### 2.6. `math.gcd` và `functools.reduce`

- `math.gcd(a, b)` là bản có sẵn, nhanh. Từ Python 3.9, `math.gcd(*nums)` nhận nhiều số.
- `functools.reduce(f, [x1, x2, x3])` tính `f(f(x1, x2), x3)` — "gộp dần" một danh sách bằng hàm 2 tham số.

Tính chất quan trọng: `gcd(a, b, c) = gcd(gcd(a, b), c)` — nên gộp dần là đúng.

In [7]:
import math
from functools import reduce

print(math.gcd(4, 6))
print(math.gcd(4, 6, 10))                 # Python 3.9+
print(reduce(math.gcd, [4, 6, 10]))       # gcd(gcd(4,6),10) = gcd(2,10) = 2
print(reduce(lambda acc, x: acc + x, [1, 2, 3, 4]))  # reduce cũng gộp được phép cộng: 10

2
2
2
10


`lambda acc, x: acc + x` là **hàm vô danh** viết trên một dòng: nhận `acc`, `x`, trả `acc + x`. Ở đây chỉ để minh họa `reduce`; với gcd ta truyền thẳng `math.gcd`.

## 3. Cách trực giác (baseline) — thử mọi kích thước `X`

Người mới thường nghĩ: "Mình chưa biết `X` là bao nhiêu, vậy **thử hết**".
- `X` phải ≥ 2.
- `X` không thể lớn hơn tổng số lá `n`.
- Với mỗi `X`, kiểm tra: **mọi** tần suất `cnt` đều chia hết cho `X`?

Nếu có một `X` thỏa → `True`. Thử hết mà không có → `False`.

In [8]:
from collections import Counter

def has_groups_bruteforce(deck: list[int]) -> bool:
    counts = Counter(deck).values()
    n = len(deck)
    for X in range(2, n + 1):
        if all(cnt % X == 0 for cnt in counts):
            return True
    return False

print(has_groups_bruteforce([1, 2, 3, 4, 4, 3, 2, 1]))  # True
print(has_groups_bruteforce([1, 1, 1, 2, 2, 2, 3, 3]))  # False
print(has_groups_bruteforce([1]))                       # False
print(has_groups_bruteforce([1, 1, 2, 2, 2, 2]))        # True

True
False
False
True


### 3.1. Đếm số phép kiểm tra của baseline

Gọi `n = len(deck)` và `k` = số giá trị khác nhau. Trường hợp xấu (đáp án `False`) ta thử `n - 1` giá trị `X`, mỗi lần kiểm tra tối đa `k` tần suất → khoảng `O(n * k)`.

Với `n = 10^4` và `k` lớn, đây vẫn chạy được, nhưng ta muốn hiểu **bản chất toán** để có lời giải gọn hơn.

In [9]:
def count_checks_bruteforce(deck):
    counts = list(Counter(deck).values())
    checks = 0
    for X in range(2, len(deck) + 1):
        for cnt in counts:
            checks += 1
            if cnt % X != 0:
                break
        else:
            return True, checks
    return False, checks

# Trường hợp xấu: 3000 số khác nhau, số 0 xuất hiện 2 lần, còn lại 1 lần -> False, phải thử hết X
deck_bad = list(range(3000)) + [0]
print(count_checks_bruteforce(deck_bad))

(False, 3001)


Cú pháp `for ... else`: khối `else` chạy khi vòng `for` **kết thúc bình thường** (không bị `break`). Ở đây nghĩa là "không tần suất nào vi phạm" → `X` hợp lệ.

## 4. Nút thắt và insight — từ "thử X" sang ƯCLN

**Nút thắt:** baseline thử rất nhiều `X` vô ích. Ta cần biết ngay "có tồn tại `X ≥ 2` chia hết mọi tần suất không?".

**Quan sát:**
- `X` chia hết mọi tần suất ⇔ `X` là **ước chung** của mọi tần suất.
- Mọi ước chung đều là ước của **ước chung lớn nhất** `g = gcd(c1, c2, ..., ck)`.
- Vậy tồn tại ước chung `X ≥ 2` ⇔ `g ≥ 2` (lấy luôn `X = g`).

**Kết luận:** đáp án = `gcd(tất cả tần suất) >= 2`.

Kiểm tra trên ví dụ bằng tay:

| deck | tần suất | gcd | đáp án |
|---|---|:---:|:---:|
| `[1,2,3,4,4,3,2,1]` | 2,2,2,2 | 2 | True |
| `[1,1,1,2,2,2,3,3]` | 3,3,2 | 1 | False |
| `[1,1,2,2,2,2]` | 2,4 | 2 | True |
| `[1]` | 1 | 1 | False |

In [10]:
import math
for deck in [[1,2,3,4,4,3,2,1], [1,1,1,2,2,2,3,3], [1,1,2,2,2,2], [1]]:
    freqs = list(Counter(deck).values())
    print(f"{str(deck):28} tần suất={freqs} gcd={math.gcd(*freqs)}")

[1, 2, 3, 4, 4, 3, 2, 1]     tần suất=[2, 2, 2, 2] gcd=2
[1, 1, 1, 2, 2, 2, 3, 3]     tần suất=[3, 3, 2] gcd=1
[1, 1, 2, 2, 2, 2]           tần suất=[2, 4] gcd=2
[1]                          tần suất=[1] gcd=1


### 4.1. Một phản ví dụ cho ý tưởng sai "lấy tần suất nhỏ nhất"

Nhiều người nghĩ: "X = tần suất nhỏ nhất". Sai! Với tần suất `[4, 6]`, nhỏ nhất là 4 nhưng 6 không chia hết cho 4; tuy vậy `X = 2` vẫn hợp lệ. Cần **gcd**, không phải **min**.

In [11]:
freqs = [4, 6]
m = min(freqs)
print("min =", m, "-> mọi tần suất chia hết cho min?", all(f % m == 0 for f in freqs))
print("gcd =", math.gcd(*freqs), "-> đáp án đúng là True")

min = 4 -> mọi tần suất chia hết cho min? False
gcd = 2 -> đáp án đúng là True


## 5. Lời giải chính — đếm tần suất rồi gộp gcd

Ta xây từng đoạn:

**Bước 1 — đếm:** `counts = Counter(deck)`.

**Bước 2 — gộp gcd:** bắt đầu `g = 0` (vì `gcd(0, c) = c`), duyệt từng tần suất: `g = gcd(g, c)`.

**Bước 3 — dừng sớm (tùy chọn):** nếu `g == 1` thì không thể quay lại ≥ 2 nữa (gcd chỉ giảm hoặc giữ nguyên) → trả `False` ngay.

**Bước 4 — kết luận:** `return g >= 2`.

In [12]:
# Kiểm tra mảnh nhỏ: vì sao khởi tạo g = 0 là an toàn?
print(math.gcd(0, 5), math.gcd(0, 12))   # gcd(0, c) = c

5 12


In [13]:
def has_groups_gcd(deck: list[int]) -> bool:
    counts = Counter(deck)           # Bước 1
    g = 0
    for cnt in counts.values():      # Bước 2
        g = math.gcd(g, cnt)
        if g == 1:                   # Bước 3: dừng sớm
            return False
    return g >= 2                    # Bước 4

for deck in [[1,2,3,4,4,3,2,1], [1,1,1,2,2,2,3,3], [1], [1,1], [1,1,2,2,2,2]]:
    print(deck, "->", has_groups_gcd(deck))

[1, 2, 3, 4, 4, 3, 2, 1] -> True
[1, 1, 1, 2, 2, 2, 3, 3] -> False
[1] -> False
[1, 1] -> True
[1, 1, 2, 2, 2, 2] -> True


## 6. TRACE từng bước

Ví dụ `deck = [1,1,2,2,2,2,3,3,3,3,3,3]` → tần suất `{1: 2, 2: 4, 3: 6}`.

| Bước | value | cnt | g trước | g sau = gcd(g, cnt) | Ghi chú |
|:---:|:---:|:---:|:---:|:---:|---|
| 1 | 1 | 2 | 0 | 2 | gcd(0,2)=2 |
| 2 | 2 | 4 | 2 | 2 | 4 chia hết cho 2 |
| 3 | 3 | 6 | 2 | 2 | 6 chia hết cho 2 |
| kết | | | | 2 | `2 >= 2` → True, X = 2 |

Ví dụ thứ hai `deck = [1,1,1,2,2,2,3,3]` → tần suất `{1: 3, 2: 3, 3: 2}`: g đi `0 → 3 → 3 → 1` và dừng sớm ở 1.

In [14]:
def trace_has_groups(deck):
    counts = Counter(deck)
    print("deck =", deck)
    print("counts =", dict(counts))
    g = 0
    for step, (value, cnt) in enumerate(counts.items(), start=1):
        new_g = math.gcd(g, cnt)
        print(f"  bước {step}: value={value}, cnt={cnt}, g: {g} -> {new_g}")
        g = new_g
        if g == 1:
            print("  g = 1 -> dừng sớm, trả False")
            return False
    print(f"  kết thúc: g = {g} -> {'True' if g >= 2 else 'False'}")
    return g >= 2

trace_has_groups([1,1,2,2,2,2,3,3,3,3,3,3])
print()
trace_has_groups([1,1,1,2,2,2,3,3])
print()
trace_has_groups([5])

deck = [1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3]
counts = {1: 2, 2: 4, 3: 6}
  bước 1: value=1, cnt=2, g: 0 -> 2
  bước 2: value=2, cnt=4, g: 2 -> 2
  bước 3: value=3, cnt=6, g: 2 -> 2
  kết thúc: g = 2 -> True

deck = [1, 1, 1, 2, 2, 2, 3, 3]
counts = {1: 3, 2: 3, 3: 2}
  bước 1: value=1, cnt=3, g: 0 -> 3
  bước 2: value=2, cnt=3, g: 3 -> 3
  bước 3: value=3, cnt=2, g: 3 -> 1
  g = 1 -> dừng sớm, trả False

deck = [5]
counts = {5: 1}
  bước 1: value=5, cnt=1, g: 0 -> 1
  g = 1 -> dừng sớm, trả False


False

`enumerate(..., start=1)` trả về cặp `(số thứ tự, phần tử)`, đếm từ 1. `counts.items()` trả các cặp `(key, value)`, nên ta viết `(value, cnt)` để **mở gói** ngay trong vòng lặp.

### 6.1. Trace bên trong thuật toán Euclid

Với tần suất `[12, 18, 8]`, hãy xem `gcd` được tính thế nào từng phép chia dư.

In [15]:
g = 0
for cnt in [12, 18, 8]:
    print(f"gộp {cnt} vào g={g}:")
    g = my_gcd(g, cnt) if g else cnt
    print("  => g =", g)

gộp 12 vào g=0:
  => g = 12
gộp 18 vào g=12:
  gcd(12, 18) -> 12 % 18 = 12
  gcd(18, 12) -> 18 % 12 = 6
  gcd(12, 6) -> 12 % 6 = 0
  => g = 6
gộp 8 vào g=6:
  gcd(6, 8) -> 6 % 8 = 6
  gcd(8, 6) -> 8 % 6 = 2
  gcd(6, 2) -> 6 % 2 = 0
  => g = 2


## 7. Các lời giải khác và bảng so sánh

Bài nhỏ nhưng có nhiều cách nhìn. Ta xem 4 cách, từ cơ bản lên:

### 7.1. Cách A — Thử mọi X (baseline, mục 3)

Đã có ở `has_groups_bruteforce`.

### 7.2. Cách B — Chỉ thử X từ 2 đến tần suất nhỏ nhất

`X` phải chia hết **mọi** tần suất, nên `X <= min(tần suất)`. Cắt bớt phạm vi thử. Hơn nữa `X` phải chia hết `n` (tổng các tần suất).

In [16]:
def has_groups_min_bound(deck: list[int]) -> bool:
    counts = list(Counter(deck).values())
    n = len(deck)
    for X in range(2, min(counts) + 1):
        if n % X != 0:          # lọc nhanh: X phải chia hết tổng số lá
            continue
        if all(cnt % X == 0 for cnt in counts):
            return True
    return False

for deck in [[1,2,3,4,4,3,2,1], [1,1,1,2,2,2,3,3], [1], [1,1,2,2,2,2]]:
    print(deck, "->", has_groups_min_bound(deck))

[1, 2, 3, 4, 4, 3, 2, 1] -> True
[1, 1, 1, 2, 2, 2, 3, 3] -> False
[1] -> False
[1, 1, 2, 2, 2, 2] -> True


### 7.3. Cách C — `reduce(math.gcd, ...)` một dòng

Cùng ý với lời giải chính nhưng viết gọn. Không có dừng sớm, nhưng vẫn `O(n)` vì phần đếm chiếm chủ yếu.

In [17]:
def has_groups_reduce(deck: list[int]) -> bool:
    return reduce(math.gcd, Counter(deck).values()) >= 2

for deck in [[1,2,3,4,4,3,2,1], [1,1,1,2,2,2,3,3], [1], [1,1,2,2,2,2]]:
    print(deck, "->", has_groups_reduce(deck))

[1, 2, 3, 4, 4, 3, 2, 1] -> True
[1, 1, 1, 2, 2, 2, 3, 3] -> False
[1] -> False
[1, 1, 2, 2, 2, 2] -> True


### 7.4. Cách D — Không dùng `Counter`: sắp xếp rồi đếm "đoạn liên tiếp"

Sau khi `sorted(deck)`, các số giống nhau nằm cạnh nhau thành **đoạn** (run). Độ dài mỗi đoạn chính là tần suất. Cách này dạy kỹ thuật **đếm run** — dùng nhiều ở bài chuỗi (ví dụ LeetCode 925 Long Pressed Name).

In [18]:
def has_groups_sort_runs(deck: list[int]) -> bool:
    arr = sorted(deck)
    g = 0
    i = 0
    while i < len(arr):
        j = i
        while j < len(arr) and arr[j] == arr[i]:
            j += 1
        run_len = j - i          # độ dài đoạn arr[i:j]
        g = math.gcd(g, run_len)
        i = j                    # nhảy sang đoạn tiếp theo
    return g >= 2

for deck in [[1,2,3,4,4,3,2,1], [1,1,1,2,2,2,3,3], [1], [1,1,2,2,2,2]]:
    print(deck, "->", has_groups_sort_runs(deck))

[1, 2, 3, 4, 4, 3, 2, 1] -> True
[1, 1, 1, 2, 2, 2, 3, 3] -> False
[1] -> False
[1, 1, 2, 2, 2, 2] -> True


### 7.5. Bảng so sánh

`n = len(deck)`, `k` = số giá trị khác nhau, `m = min tần suất`, `C = max(deck)`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| A. Thử mọi X | `X = 2..n`, kiểm tra all | `O(n·k)` | `O(k)` | dễ nghĩ nhất |
| B. Giới hạn X ≤ min | `X = 2..m`, lọc `n % X` | `O(n + m·k)` | `O(k)` | nhanh hơn A |
| **Chính: gcd + dừng sớm** | gộp gcd các tần suất | `O(n + k·log C')` ≈ `O(n)` | `O(k)` | nên nộp |
| C. reduce gcd | một dòng | ≈ `O(n)` | `O(k)` | ngắn nhất |
| D. sort + run | đếm đoạn liên tiếp | `O(n log n)` | `O(n)` | không cần Counter |

(`C'` ≤ `n` là giá trị tần suất; mỗi lần `gcd` tốn `O(log n)` bước chia dư.)

## 8. Code nộp LeetCode

Bản chính (gcd + dừng sớm). LeetCode đã import sẵn `math`, `collections`, nhưng ta vẫn ghi `import` cho rõ — an toàn khi chạy ở nơi khác.

In [19]:
import math
from collections import Counter

class Solution:
    def hasGroupsSizeX(self, deck: list[int]) -> bool:
        g = 0
        for cnt in Counter(deck).values():
            g = math.gcd(g, cnt)
            if g == 1:
                return False
        return g >= 2

sol = Solution()
assert sol.hasGroupsSizeX([1,2,3,4,4,3,2,1]) == True
assert sol.hasGroupsSizeX([1,1,1,2,2,2,3,3]) == False
print("Solution chạy đúng 2 ví dụ đề bài")

Solution chạy đúng 2 ví dụ đề bài


Bản ngắn (cách C) cũng được chấp nhận:

```python
class Solution:
    def hasGroupsSizeX(self, deck: list[int]) -> bool:
        return reduce(math.gcd, Counter(deck).values()) >= 2
```

## 9. Vì sao thuật toán đúng?

Gọi các tần suất là `c1, c2, ..., ck` và `g = gcd(c1, ..., ck)`.

**Chiều ⇐ (g ≥ 2 thì chia được):** chọn `X = g`. Mỗi giá trị `i` có `ci` lá, `ci` chia hết cho `g` nên chia thành `ci / g` nhóm, mỗi nhóm `g` lá cùng số. Mọi lá đều được dùng → hợp lệ.

**Chiều ⇒ (chia được thì g ≥ 2):** giả sử chia được với kích thước `X ≥ 2`. Mỗi nhóm chỉ chứa một giá trị, nên các lá của giá trị `i` bị chia hết vào các nhóm kích thước `X` → `X | ci` với mọi `i`. Vậy `X` là ước chung, nên `X | g`, suy ra `g ≥ X ≥ 2`.

**Invariant của vòng lặp:** sau khi duyệt `j` tần suất đầu, `g = gcd(c1, ..., cj)`. Khởi tạo `g = 0` đúng vì `gcd(0, c1) = c1`. Mỗi bước `gcd(g, c_{j+1})` giữ nguyên invariant nhờ `gcd(a, b, c) = gcd(gcd(a, b), c)`.

**Dừng sớm đúng** vì `gcd(1, x) = 1` với mọi `x`: một khi `g = 1`, kết quả cuối chắc chắn là 1 → `False`.

In [20]:
# Kiểm chứng chiều <=: dựng nhóm thật với X = g và kiểm tra hợp lệ
def build_groups(deck):
    counts = Counter(deck)
    g = reduce(math.gcd, counts.values())
    if g < 2:
        return None
    groups = []
    for value, cnt in counts.items():
        for _ in range(cnt // g):
            groups.append([value] * g)
    return groups

groups = build_groups([3,3,3,3,7,7,7,7,7,7])
print(groups)
assert all(len(gr) == len(groups[0]) >= 2 and len(set(gr)) == 1 for gr in groups)

[[3, 3], [3, 3], [7, 7], [7, 7], [7, 7]]


## 10. Phân tích Time / Space

Gọi `n = len(deck)`, `k` = số giá trị khác nhau (`k ≤ n`).

- **Đếm** bằng `Counter`: duyệt `n` phần tử, mỗi lần cập nhật dict trung bình `O(1)` → `O(n)`.
- **Gộp gcd:** `k` lần gọi `gcd`, mỗi lần `O(log n)` (Euclid giảm số ít nhất một nửa sau 2 bước) → `O(k log n)`.
- Tổng: `O(n + k log n)` = `O(n log n)` trong trường hợp xấu nhất về lý thuyết, nhưng thực tế gần `O(n)`; nhiều tài liệu ghi `O(n log n)` hoặc `O(n)` tùy cách tính gcd.
- **Space:** `O(k)` cho `Counter` (tối đa `O(n)`).

Với baseline: `O(n · k)` — khi `n = 10^4` và `k ≈ n/2` có thể tới khoảng 10^7 phép kiểm tra.

In [21]:
import time, random
rng = random.Random(914)
big = [rng.randrange(5000) for _ in range(10000)]
for name, f in [("bruteforce", has_groups_bruteforce), ("gcd", has_groups_gcd)]:
    t0 = time.perf_counter()
    r = f(big)
    print(f"{name:10} -> {r}, {time.perf_counter() - t0:.4f}s")

bruteforce -> False, 0.0053s
gcd        -> False, 0.0007s


## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ví dụ đề bài.
2. Biên: 1 lá, 2 lá giống nhau, 2 lá khác nhau.
3. Một giá trị chiếm nhiều nhóm: `[1,1,2,2,2,2]`.
4. Phản ví dụ cho "min": tần suất `[4, 6]` → True.
5. Tần suất nguyên tố cùng nhau: `[2, 3]` → False.
6. Một giá trị duy nhất lặp nhiều lần.

In [22]:
tests = [
    ([1,2,3,4,4,3,2,1], True),
    ([1,1,1,2,2,2,3,3], False),
    ([1], False),
    ([1,1], True),
    ([1,2], False),
    ([1,1,2,2,2,2], True),
    ([1]*4 + [2]*6, True),
    ([1]*2 + [2]*3, False),
    ([0]*7, True),
    ([0]*6 + [1]*9 + [2]*12, True),
    ([0]*6 + [1]*9 + [2]*10, False),
]
solvers = [has_groups_bruteforce, has_groups_min_bound, has_groups_gcd,
           has_groups_reduce, has_groups_sort_runs, Solution().hasGroupsSizeX]
for deck, expected in tests:
    for f in solvers:
        got = f(deck)
        assert got == expected, (f.__name__, deck, got, expected)
print(f"PASS {len(tests)} test x {len(solvers)} cách")

PASS 11 test x 6 cách


In [23]:
# Kiểm thử ngẫu nhiên có seed: mọi cách phải trả cùng kết quả với baseline
rng = random.Random(2024)
for _ in range(500):
    k = rng.randint(1, 4)
    deck = []
    for v in range(k):
        deck += [v] * rng.randint(1, 12)
    rng.shuffle(deck)
    expected = has_groups_bruteforce(deck)
    for f in solvers[1:]:
        assert f(deck) == expected, (f.__name__, deck)
print("PASS 500 test ngẫu nhiên")

PASS 500 test ngẫu nhiên


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng tần suất nhỏ nhất làm X

In [24]:
def wrong_min(deck):
    counts = list(Counter(deck).values())
    X = min(counts)
    return X >= 2 and all(c % X == 0 for c in counts)

print("wrong_min([1]*4+[2]*6) =", wrong_min([1]*4+[2]*6), "| đúng phải là True")

wrong_min([1]*4+[2]*6) = False | đúng phải là True


**Vì sao sai:** `X` hợp lệ chỉ cần là **ước chung**, không nhất thiết bằng tần suất nhỏ nhất. **Sửa:** dùng gcd.

### Lỗi 2: Nghĩ mỗi giá trị chỉ được tạo đúng một nhóm (so sánh tần suất bằng nhau)

In [25]:
def wrong_all_equal(deck):
    counts = list(Counter(deck).values())
    return counts[0] >= 2 and len(set(counts)) == 1

print("wrong_all_equal([1,1,2,2,2,2]) =", wrong_all_equal([1,1,2,2,2,2]), "| đúng phải là True")

wrong_all_equal([1,1,2,2,2,2]) = False | đúng phải là True


**Vì sao sai:** đề cho phép một giá trị nằm trong **nhiều** nhóm. **Sửa:** chỉ cần mỗi tần suất chia hết cho `X`.

### Lỗi 3: Quên điều kiện X > 1 (trả `True` khi gcd = 1)

In [26]:
def wrong_ge1(deck):
    return reduce(math.gcd, Counter(deck).values()) >= 1

print("wrong_ge1([1,2]) =", wrong_ge1([1,2]), "| đúng phải là False (X=1 không được phép)")

wrong_ge1([1,2]) = True | đúng phải là False (X=1 không được phép)


**Vì sao sai:** `gcd ≥ 1` luôn đúng. **Sửa:** so sánh `>= 2`.

### Lỗi 4: Gộp gcd trên chính các giá trị trong deck thay vì tần suất

`reduce(math.gcd, deck)` tính gcd của **các số ghi trên lá bài** — không liên quan gì đến số lần xuất hiện. Ví dụ `[2,4]` có gcd giá trị là 2 nhưng mỗi số chỉ xuất hiện 1 lần → đáp án False.

In [27]:
deck = [2, 4]
print("gcd các giá trị:", reduce(math.gcd, deck), "(sai đối tượng)")
print("gcd các tần suất:", reduce(math.gcd, Counter(deck).values()), "-> False là đúng")

gcd các giá trị: 2 (sai đối tượng)
gcd các tần suất: 1 -> False là đúng


### Lỗi 5: `math.gcd(counts)` truyền cả list

`math.gcd` nhận các **số**, không nhận list. Phải viết `math.gcd(*counts)` (dấu `*` "trải" list thành nhiều tham số) hoặc dùng `reduce`.

In [28]:
counts = [4, 6, 8]
try:
    math.gcd(counts)
except TypeError as e:
    print("TypeError:", e)
print("math.gcd(*counts) =", math.gcd(*counts))

TypeError: 'list' object cannot be interpreted as an integer
math.gcd(*counts) = 2


## 13. Mẹo nhận dạng pattern

- Đề nói "chia thành các nhóm **bằng nhau**", "mỗi nhóm cùng loại" → nghĩ tới **đếm tần suất** + **ước chung**.
- "Có tồn tại kích thước chung chia hết mọi số" ⇔ **gcd ≥ 2**.
- "Thứ tự không quan trọng" → bỏ thứ tự, chỉ giữ **Counter**.
- Khi một điều kiện phải đúng "cho mọi phần tử" và có tính chất gộp được (gcd, min, max, and) → dùng **reduce/gộp dần** và cân nhắc **dừng sớm** khi đạt giá trị "hấp thụ" (gcd = 1, and = False...).

Bài liên quan: 1071 Greatest Common Divisor of Strings, 1979 Find GCD of Array, 2344 Minimum Deletions to Make Array Divisible, 846 Hand of Straights (chia nhóm nhưng nhóm là dãy liên tiếp).

## 14. Biến thể

### 14.1. Trả về kích thước nhóm X lớn nhất (hoặc 0 nếu không có)

Chính là `g` nếu `g ≥ 2`.

In [29]:
def largest_group_size(deck):
    g = reduce(math.gcd, Counter(deck).values())
    return g if g >= 2 else 0

print(largest_group_size([1,1,2,2,2,2]))      # 2
print(largest_group_size([0]*6 + [1]*9))      # 3
print(largest_group_size([1,2]))              # 0

2
3
0


### 14.2. Liệt kê **mọi** X hợp lệ

Mọi `X` hợp lệ là các ước ≥ 2 của `g`. Duyệt `d` từ 2 đến `g`.

In [30]:
def all_valid_sizes(deck):
    g = reduce(math.gcd, Counter(deck).values())
    return [d for d in range(2, g + 1) if g % d == 0]

print(all_valid_sizes([0]*12 + [1]*24))   # g=12 -> [2,3,4,6,12]
print(all_valid_sizes([1,2,3]))           # []

[2, 3, 4, 6, 12]
[]


### 14.3. Bắt buộc dùng đúng kích thước `X` cho trước

Chỉ cần kiểm tra `X >= 2 and all(cnt % X == 0 ...)` — không cần gcd. Đây là cách học viên dễ nhầm sang khi đề đổi nhẹ, hãy đọc kỹ đề.

In [31]:
def can_group_with_size(deck, X):
    return X >= 2 and all(c % X == 0 for c in Counter(deck).values())

print(can_group_with_size([1]*4 + [2]*6, 2), can_group_with_size([1]*4 + [2]*6, 3))

True False


## 15. Bài tập tự luyện (TODO — không tự chạy)

### Bài 1 — Đoán output (tính tay)
Không chạy code, đoán:
```python
Counter([5, 5, 7, 5, 7, 9]).values()   # ?
math.gcd(0, 9), math.gcd(12, 18), math.gcd(7, 5)   # ?
```

### Bài 2 — Tự viết gcd bằng Euclid

In [32]:
def exercise_gcd(a: int, b: int) -> int:
    # TODO: dùng vòng while và a, b = b, a % b
    pass

# assert exercise_gcd(48, 18) == 6
# assert exercise_gcd(7, 5) == 1
# assert exercise_gcd(0, 9) == 9

### Bài 3 — gcd của cả danh sách (không dùng reduce)

In [33]:
def exercise_gcd_list(nums: list[int]) -> int:
    # TODO: khởi tạo g = 0, gộp dần bằng exercise_gcd hoặc math.gcd
    pass

# assert exercise_gcd_list([12, 18, 8]) == 2
# assert exercise_gcd_list([5]) == 5

### Bài 4 — Tự viết lại lời giải LeetCode 914

In [34]:
def exercise_has_groups(deck: list[int]) -> bool:
    # TODO 1: đếm tần suất
    # TODO 2: gộp gcd, dừng sớm khi g == 1
    # TODO 3: trả g >= 2
    pass

# assert exercise_has_groups([1,2,3,4,4,3,2,1]) == True
# assert exercise_has_groups([1,1,1,2,2,2,3,3]) == False
# assert exercise_has_groups([1,1,2,2,2,2]) == True

### Bài 5 — Biến thể: số nhóm ít nhất

Nếu chia được, trả về **số nhóm ít nhất** có thể (dùng `X` lớn nhất). Nếu không chia được, trả `-1`.

In [35]:
def exercise_min_groups(deck: list[int]) -> int:
    # TODO
    pass

# assert exercise_min_groups([1,1,2,2,2,2]) == 3
# assert exercise_min_groups([0]*6 + [1]*9) == 5
# assert exercise_min_groups([1,2]) == -1

### Bài 6 — Mini challenge: thêm ít lá nhất

Cho `deck`, được phép **thêm** lá bài có số **đã xuất hiện**. Hỏi cần thêm ít nhất bao nhiêu lá để chia được với `X = 2`? (Gợi ý: chỉ quan tâm tần suất lẻ.)

In [36]:
def exercise_min_add_for_pairs(deck: list[int]) -> int:
    # TODO
    pass

# assert exercise_min_add_for_pairs([1,1,2,3,3,3]) == 2
# assert exercise_min_add_for_pairs([4,4]) == 0

## 16. Gợi ý 3 mức

| Bài | Mức 1 (hướng nghĩ) | Mức 2 (cấu trúc) | Mức 3 (gần code) |
|---|---|---|---|
| 1 | Counter giữ thứ tự xuất hiện đầu tiên | đếm 5, 7, 9 | `gcd(0,x)=x` |
| 2 | gcd(a,b) = gcd(b, a%b) | `while b:` | `a, b = b, a % b; return a` |
| 3 | gộp dần như cộng tổng | `g = 0` rồi `for x in nums` | `g = math.gcd(g, x)` |
| 4 | tần suất + gcd | `Counter(deck).values()` | `if g == 1: return False` |
| 5 | ít nhóm nhất ⇔ X lớn nhất | X = g | `len(deck) // g` nếu `g >= 2` |
| 6 | X = 2 ⇔ mọi tần suất chẵn | đếm tần suất lẻ | `sum(1 for c in ... if c % 2)` |

## 17. Đáp án tham khảo

Bài 1: `Counter([5,5,7,5,7,9]).values()` → `dict_values([3, 2, 1])`; `math.gcd(0,9)=9`, `math.gcd(12,18)=6`, `math.gcd(7,5)=1`.

In [37]:
print(Counter([5, 5, 7, 5, 7, 9]).values())
print(math.gcd(0, 9), math.gcd(12, 18), math.gcd(7, 5))

dict_values([3, 2, 1])
9 6 1


In [38]:
def answer_gcd(a: int, b: int) -> int:
    while b:
        a, b = b, a % b
    return a

def answer_gcd_list(nums: list[int]) -> int:
    g = 0
    for x in nums:
        g = answer_gcd(g, x)
    return g

def answer_has_groups(deck: list[int]) -> bool:
    g = 0
    for cnt in Counter(deck).values():
        g = answer_gcd(g, cnt)
        if g == 1:
            return False
    return g >= 2

def answer_min_groups(deck: list[int]) -> int:
    g = answer_gcd_list(list(Counter(deck).values()))
    return len(deck) // g if g >= 2 else -1

def answer_min_add_for_pairs(deck: list[int]) -> int:
    return sum(1 for c in Counter(deck).values() if c % 2 == 1)

assert answer_gcd(48, 18) == 6 and answer_gcd(7, 5) == 1 and answer_gcd(0, 9) == 9
assert answer_gcd_list([12, 18, 8]) == 2 and answer_gcd_list([5]) == 5
assert answer_has_groups([1,2,3,4,4,3,2,1]) == True
assert answer_has_groups([1,1,1,2,2,2,3,3]) == False
assert answer_has_groups([1,1,2,2,2,2]) == True
assert answer_min_groups([1,1,2,2,2,2]) == 3
assert answer_min_groups([0]*6 + [1]*9) == 5
assert answer_min_groups([1,2]) == -1
assert answer_min_add_for_pairs([1,1,2,3,3,3]) == 2
assert answer_min_add_for_pairs([4,4]) == 0
for deck, expected in tests:
    assert answer_has_groups(deck) == expected
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
import math
from collections import Counter

class Solution:
    def hasGroupsSizeX(self, deck: list[int]) -> bool:
        g = 0
        for cnt in Counter(deck).values():
            g = math.gcd(g, cnt)
            if g == 1:
                return False
        return g >= 2
```

**Câu thần chú:** *"Chia nhóm bằng nhau, mỗi nhóm một loại ⇒ gcd của các tần suất ≥ 2."*

**Checklist trước khi submit:**
- [ ] Đếm **tần suất**, không phải gcd của các giá trị lá bài.
- [ ] So sánh `g >= 2` (không phải `>= 1`).
- [ ] Khởi tạo `g = 0` (hoặc tần suất đầu tiên), không khởi tạo `g = 1`.
- [ ] Không dùng `min(tần suất)` thay gcd.
- [ ] Nhớ trường hợp `deck` có 1 lá → `False`.
- [ ] Nếu dùng `math.gcd` nhiều tham số, nhớ dấu `*`.